# Part 4: Advanced Topics & Production Patterns (Optional)
### Optional Deep Dive for Production-Ready Systems

<div style="background: linear-gradient(135deg, #581c87 0%, #7c3aed 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(255,255,255,0.1);
            padding: 28px;
            border-radius: 12px;
            color: white;
            margin: 0;
            box-shadow: 0 10px 40px rgba(88,28,135,0.4), 0 2px 12px rgba(124,58,237,0.3);">
    
<h2 style="margin: 0 0 16px 0; font-size: 1.8em; color: white; font-weight: 700;">
    ⏱️ Self-Paced: 30-45 Minutes
</h2>

<p style="margin: 0 0 12px 0; font-size: 1.1em; line-height: 1.7; opacity: 0.95;">
    Explore production patterns used in Blaize Bazaar: Aurora session management, 
    hybrid search with RRF, vector quantization, resilience patterns, and observability.
</p>

<div style="background: rgba(255,255,255,0.15);
            padding: 18px;
            border-radius: 10px;
            margin-top: 18px;">
    <p style="margin: 0 0 12px 0; font-size: 1.02em; color: white; line-height: 1.7;">
        <strong>Section 1: Aurora Session Management (8 min)</strong><br>
        ACID guarantees + Multi-tenancy patterns
    </p>
    <p style="margin: 0 0 12px 0; font-size: 1.02em; color: white; line-height: 1.7;">
        <strong>Section 2: Hybrid Search with RRF (8 min)</strong><br>
        Combining semantic + keyword search
    </p>
    <p style="margin: 0 0 12px 0; font-size: 1.02em; color: white; line-height: 1.7;">
        <strong>Section 3: Vector Quantization (8 min)</strong><br>
        Binary + Scalar quantization for scale
    </p>
    <p style="margin: 0 0 12px 0; font-size: 1.02em; color: white; line-height: 1.7;">
        <strong>Section 4: Resilience Patterns (8 min)</strong><br>
        Circuit breakers + Exponential backoff
    </p>
    <p style="margin: 0; font-size: 1.02em; color: white; line-height: 1.7;">
        <strong>Section 5: Observability (8 min)</strong><br>
        Cost tracking + OpenTelemetry tracing
    </p>
</div>

<p style="margin: 18px 0 0 0; font-size: 0.98em; color: white; opacity: 0.9;">
    <strong>🎯 Note:</strong> This is an optional deep dive. Parts 1-3 cover the core workshop content.
</p>

</div>

---

## Workshop Recap

**What you've built in Parts 1-3:**

✅ **Part 1:** Semantic search with pgvector + Aurora PostgreSQL  
✅ **Part 2:** Custom tools with 96% token reduction  
✅ **Part 3:** Multi-agent orchestration with "Agents as Tools" pattern

**Part 4 explores:** Production patterns that power Blaize Bazaar at scale.

---

## ⚙️ Step 0: Select Python Kernel

<div style="background: linear-gradient(135deg, rgba(245,158,11,0.18) 0%, rgba(217,119,6,0.18) 100%);
            backdrop-filter: blur(10px);
            border: 1px solid rgba(245,158,11,0.35);
            border-left: 6px solid #f59e0b;
            padding: 20px;
            border-radius: 10px;
            margin: 20px 0;
            box-shadow: 0 4px 16px rgba(245,158,11,0.25);">
    <h4 style="margin: 0 0 15px 0; color: #fbbf24; font-size: 1.2em; font-weight: 600;">
        ⚠️ IMPORTANT FIRST STEP
    </h4>
    <p style="margin: 0; color: #e9ecef; line-height: 1.6; font-size: 1.02em;">
        Before running any code, ensure you're using <strong style="color: #fbbf24;">Python 3.13</strong> kernel.
    </p>
</div>

## Setup: Environment & Database Connection

In [ ]:
# ============================================================================
# Environment Setup
# ============================================================================

import os
import json
import time
import psycopg
from psycopg.rows import dict_row
import boto3
from typing import List, Dict, Optional
from dotenv import load_dotenv
from datetime import datetime
import numpy as np

load_dotenv('/workshop/sample-dat406-build-agentic-ai-powered-search-apg/.env')

# Database configuration
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_NAME = os.getenv('DB_NAME', 'postgres')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

conn_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
bedrock = boto3.client('bedrock-runtime', region_name=os.environ.get('AWS_REGION', 'us-west-2'))

print("✅ Environment configured for advanced patterns")
print(f"   Database: {DB_NAME}")
print(f"   Region: {bedrock.meta.region_name}")

---

# Section 1: Aurora Session Management

## Why Aurora for Sessions?

**The challenge:** Multi-agent systems need to maintain conversation state across multiple agent invocations. Where do you store this state?

**Common approaches:**
- ❌ **Redis/ElastiCache** - Fast but requires separate infrastructure, no ACID guarantees
- ❌ **DynamoDB** - Scalable but eventual consistency, complex queries
- ✅ **Aurora PostgreSQL** - Same database as your vector search, ACID guarantees, rich querying

**Production requirements:**
- **ACID guarantees** - Consistent conversation state across concurrent requests
- **Multi-tenancy** - Isolated customer sessions with proper data boundaries
- **Audit trail** - Complete interaction history for debugging and compliance
- **Same database** - No separate session store needed, reduced latency and complexity

**Blaize Bazaar's approach:**

## AuroraSessionManager Implementation

**Key features:**
- Automatic session creation and retrieval
- Message persistence with timestamps
- Tool use tracking for debugging
- Conversation metadata (agent_name, context, metadata)

### Session State Architecture

<div style="text-align: center; margin: 20px 0;">
    <img src="diagrams/part4_session_state_diagram.svg" alt="Aurora Session State Diagram" style="max-width: 100%; border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.3);" />
</div>

*Four-table schema with CASCADE deletes ensures ACID-guaranteed conversation state with complete audit trail*

**Schema:**
```sql
CREATE TABLE conversations (
    session_id VARCHAR(255) PRIMARY KEY,
    agent_name VARCHAR(255),
    context JSONB DEFAULT '{}'::jsonb,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    metadata JSONB DEFAULT '{}'::jsonb
);

CREATE TABLE messages (
    id SERIAL PRIMARY KEY,
    session_id VARCHAR(255) REFERENCES conversations(session_id),
    role VARCHAR(50) CHECK (role IN ('user', 'assistant', 'system')),
    content TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    metadata JSONB DEFAULT '{}'::jsonb
);

CREATE TABLE tool_uses (
    id SERIAL PRIMARY KEY,
    session_id VARCHAR(255) REFERENCES conversations(session_id),
    tool_name VARCHAR(255) NOT NULL,
    tool_input JSONB,
    tool_output JSONB,
    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

In [ ]:
# ============================================================================
# AuroraSessionManager - Production Session Management
# ============================================================================

class AuroraSessionManager:
    """Session management with Aurora PostgreSQL for ACID guarantees."""
    
    def __init__(self, conn_string: str):
        self.conn_string = conn_string
    
    def create_session(self, agent_name: str = None) -> str:
        """Create new conversation session."""
        import uuid
        session_id = str(uuid.uuid4())
        
        with psycopg.connect(self.conn_string) as conn:
            with conn.cursor() as cur:
                cur.execute("""
                    INSERT INTO bedrock_integration.conversations 
                    (session_id, agent_name)
                    VALUES (%s, %s)
                """, (session_id, agent_name))
                conn.commit()
        
        return session_id
    
    def add_message(self, session_id: str, role: str, content: str):
        """Add message to conversation history."""
        with psycopg.connect(self.conn_string) as conn:
            with conn.cursor() as cur:
                cur.execute("""
                    INSERT INTO bedrock_integration.messages 
                    (session_id, role, content)
                    VALUES (%s, %s, %s)
                """, (session_id, role, content))
                
                # Update conversation timestamp
                cur.execute("""
                    UPDATE bedrock_integration.conversations 
                    SET updated_at = NOW() WHERE session_id = %s
                """, (session_id,))
                
                conn.commit()
    
    def get_conversation_history(self, session_id: str) -> List[Dict]:
        """Retrieve full conversation history."""
        with psycopg.connect(self.conn_string) as conn:
            with conn.cursor(row_factory=dict_row) as cur:
                cur.execute("""
                    SELECT role, content, created_at 
                    FROM bedrock_integration.messages
                    WHERE session_id = %s
                    ORDER BY created_at ASC
                """, (session_id,))
                return [dict(row) for row in cur.fetchall()]

print("✅ AuroraSessionManager class defined")
print("   Features: ACID guarantees, multi-tenancy, audit trail")

**Benefits:**
- ✅ Transactional consistency across messages
- ✅ No separate session store (Redis/DynamoDB) needed
- ✅ Same database as vector search (reduced latency)
- ✅ Full audit trail for compliance

---

# Section 2: Hybrid Search with Reciprocal Rank Fusion

## The RRF Algorithm

**Challenge:** Semantic search alone misses exact keyword matches.

**Real-world example:** User searches "wireless bluetooth headphones"
- Semantic search finds: "Premium audio device with wireless connectivity" ✅
- But misses: Product titled exactly "Wireless Bluetooth Headphones" ❌

**Solution:** Combine semantic (pgvector) + keyword (PostgreSQL FTS)

**Reciprocal Rank Fusion (RRF):**
```
score(doc) = Σ 1 / (k + rank_in_result_i)

Where:
- k = 60 (constant, prevents division by zero)
- rank = position in each result set (1, 2, 3...)
```

**Why RRF over weighted average?**
- ✅ No manual weight tuning needed
- ✅ Robust to score scale differences (semantic: 0-1, keyword: 0-100)
- ✅ Used by Elasticsearch, OpenSearch in production

**Example:**
```
Semantic results: [DocA(rank=1), DocB(rank=2), DocC(rank=3)]
Keyword results:  [DocB(rank=1), DocD(rank=2), DocA(rank=3)]

RRF scores:
- DocA: 1/(60+1) + 1/(60+3) = 0.0328 ← Wins! (strong in both)
- DocB: 1/(60+2) + 1/(60+1) = 0.0325  
- DocC: 1/(60+3) = 0.0159 ← Only in semantic
- DocD: 1/(60+2) = 0.0161 ← Only in keyword

Final ranking: DocA > DocB > DocD > DocC
```

In [ ]:
# ============================================================================
# Hybrid Search with Reciprocal Rank Fusion
# ============================================================================

def generate_embedding(text: str) -> List[float]:
    """Generate embeddings using Bedrock."""
    response = bedrock.invoke_model(
        modelId="amazon.titan-embed-text-v2:0",
        body=json.dumps({"inputText": text})
    )
    return json.loads(response['body'].read())['embedding']


def hybrid_search_rrf(query: str, limit: int = 5, k: int = 60) -> List[Dict]:
    """
    Hybrid search combining semantic (pgvector) and keyword (FTS) with RRF.
    
    Args:
        query: Search query
        limit: Number of results
        k: RRF constant (default: 60)
    
    Returns:
        List of products ranked by RRF score
    """
    # Generate embedding for semantic search
    query_embedding = generate_embedding(query)
    
    with psycopg.connect(conn_string) as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            # Semantic search results
            cur.execute("""
                SELECT "productId", product_description, price, stars,
                       ROW_NUMBER() OVER (ORDER BY embedding <=> %s::vector) as semantic_rank
                FROM bedrock_integration.product_catalog
                WHERE quantity > 0
                ORDER BY embedding <=> %s::vector
                LIMIT 20
            """, (str(query_embedding), str(query_embedding)))
            semantic_results = {r["productId"]: r for r in cur.fetchall()}
            
            # Keyword search results (PostgreSQL Full-Text Search)
            cur.execute("""
                SELECT "productId", product_description, price, stars,
                       ROW_NUMBER() OVER (ORDER BY ts_rank(to_tsvector('english', product_description), 
                                         plainto_tsquery('english', %s)) DESC) as keyword_rank
                FROM bedrock_integration.product_catalog
                WHERE quantity > 0 
                  AND to_tsvector('english', product_description) @@ plainto_tsquery('english', %s)
                ORDER BY ts_rank(to_tsvector('english', product_description), 
                                plainto_tsquery('english', %s)) DESC
                LIMIT 20
            """, (query, query, query))
            keyword_results = {r["productId"]: r for r in cur.fetchall()}
            
            # Compute RRF scores
            rrf_scores = {}
            all_products = set(semantic_results.keys()) | set(keyword_results.keys())
            
            for product_id in all_products:
                score = 0.0
                
                # Add semantic contribution
                if product_id in semantic_results:
                    rank = semantic_results[product_id]["semantic_rank"]
                    score += 1.0 / (k + rank)
                
                # Add keyword contribution
                if product_id in keyword_results:
                    rank = keyword_results[product_id]["keyword_rank"]
                    score += 1.0 / (k + rank)
                
                rrf_scores[product_id] = score
            
            # Sort by RRF score and get top results
            sorted_products = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:limit]
            
            # Fetch full product details
            product_ids = [pid for pid, _ in sorted_products]
            placeholders = ','.join(['%s'] * len(product_ids))
            
            cur.execute(f"""
                SELECT "productId", product_description, price, stars, reviews, 
                       category_name, "imgUrl", "productURL"
                FROM bedrock_integration.product_catalog
                WHERE "productId" IN ({placeholders})
            """, product_ids)
            
            products_map = {r["productId"]: dict(r) for r in cur.fetchall()}
            
            # Build final results with RRF scores
            results = []
            for product_id, rrf_score in sorted_products:
                if product_id in products_map:
                    product = products_map[product_id]
                    product['rrf_score'] = round(rrf_score, 4)
                    product['in_semantic'] = product_id in semantic_results
                    product['in_keyword'] = product_id in keyword_results
                    results.append(product)
            
            return results

# Test hybrid search
print("✅ Hybrid search with RRF implemented")
print("\nTesting: 'wireless bluetooth headphones'")
results = hybrid_search_rrf("wireless bluetooth headphones", limit=3)

for i, product in enumerate(results, 1):
    print(f"\n{i}. {product['product_description'][:60]}")
    print(f"   RRF Score: {product['rrf_score']:.4f}")
    print(f"   In semantic: {product['in_semantic']} | In keyword: {product['in_keyword']}")
    print(f"   Price: ${float(product['price']):.2f} | Rating: {float(product['stars'])}⭐")

<div style="background: linear-gradient(135deg, rgba(88,28,135,0.18) 0%, rgba(124,58,237,0.12) 100%);
            backdrop-filter: blur(8px);
            border: 1px solid rgba(124,58,237,0.3);
            border-left: 5px solid #7c3aed;
            padding: 18px 20px;
            border-radius: 8px;
            margin: 16px 0;
            box-shadow: 0 3px 10px rgba(88,28,135,0.25);">
    <strong style="color: #c4b5fd; font-size: 1.05em; display: block; margin-bottom: 8px;">
        🔧 Try It in Blaize Bazaar
    </strong>
    <span style="color: #e9d5ff; font-size: 0.95em; line-height: 1.6;">
        Hybrid search with RRF is available in the Developer Toolkit on the Blaize Bazaar app. 
        Test semantic vs keyword vs hybrid search side-by-side with real product data.
    </span>
</div>


**Why RRF works:**
- ✅ Balances semantic understanding with exact keyword matches
- ✅ No manual weight tuning needed (unlike weighted combinations)
- ✅ Robust to differences in score scales between methods
- ✅ Used by production search systems (Elasticsearch, OpenSearch)

---

# Section 3: Vector Quantization at Scale

## Why Quantization Matters

**The scaling challenge:** As your product catalog grows, vector indexes become expensive.

**Blaize Bazaar's numbers:**
- 21,704 products × 1024-dim float32 vectors = **84 MB** just for embeddings
- At 1M products: **4 GB** of vector data
- At 10M products: **40 GB** of vector data

**Impact on production:**
- 💰 **Storage costs** - Larger Aurora instances needed
- 🐌 **Query latency** - More I/O to read vectors from disk
- 📉 **ANN performance** - HNSW index traversal slows with larger indexes

**Solution:** Quantization reduces vector size while maintaining search quality.

---

## Quantization Options in pgvector 0.8.0+

| Technique | Storage per Vector | Size Reduction | Recall Impact | Use Case |
|-----------|-------------------|----------------|---------------|----------|
| **float32** (vector) | 4,096 bytes | Baseline | 100% | Maximum precision |
| **float16** (halfvec) | 2,048 bytes | **50%** | ~99%+ | **Production default** |
| **binary** (bit) | 128 bytes | **97%** | ~90-95%* | Massive scale + reranking |

*Binary recall improves significantly with two-stage retrieval (reranking)

---

## Scalar Quantization (halfvec) — Production Recommended

**Challenge:** 1024-dim float32 vectors = 4KB per vector

**Solution:** halfvec uses 16-bit floats = 2KB per vector (50% reduction)

**How it works:**
```
Original float32: 3.14159265... (32 bits = 4 bytes per dimension)
halfvec float16:  3.14159...    (16 bits = 2 bytes per dimension)

Rule: Reduce precision, not information
      The least significant digits rarely affect nearest neighbor results
```

**Trade-offs:**
- ✅ **50% smaller indexes** (4KB → 2KB per vector)
- ✅ **No two-stage retrieval needed** — query directly
- ✅ **~99%+ recall** — negligible precision loss for similarity search
- ✅ **Faster index builds** — 2x speedup on large datasets
- ✅ **Supports up to 4,000 dimensions** (vs 2,000 for float32)

**pgvector implementation:**

```sql
-- Option A: Store vectors as halfvec (permanent conversion)
ALTER TABLE products ADD COLUMN embedding_half halfvec(1024);
UPDATE products SET embedding_half = embedding::halfvec(1024);

-- Create HNSW index on halfvec column
CREATE INDEX idx_products_halfvec ON products 
    USING hnsw (embedding_half halfvec_cosine_ops)
    WITH (m = 16, ef_construction = 128);

-- Query directly — no reranking needed
SELECT product_id, product_description, price
FROM products
ORDER BY embedding_half <=> '[...]'::halfvec(1024)
LIMIT 10;


-- Option B: Index quantized, store full precision (best of both worlds)
-- Keep original float32 vectors for maximum precision when needed
-- Index uses halfvec for faster searches
CREATE INDEX idx_products_halfvec_cast ON products 
    USING hnsw ((embedding::halfvec(1024)) halfvec_cosine_ops)
    WITH (m = 16, ef_construction = 128);

-- Query requires cast to match index
SELECT product_id, product_description, price
FROM products
ORDER BY embedding::halfvec(1024) <=> '[...]'::halfvec(1024)
LIMIT 10;
```

**When to use halfvec:**
- ✅ Default choice for most production workloads
- ✅ Up to 1M vectors where recall matters
- ✅ When you can't afford two-stage retrieval complexity

---

## Binary Quantization — Maximum Compression

**Challenge:** Even halfvec may be too large at massive scale (10M+ vectors)

**Solution:** Binary quantization = 128 bytes per vector (97% reduction from float32)

**How it works:**
```
Original: [-0.5, 0.3, -0.1, 0.8, ...] (1024 floats)
Binary:   [0, 1, 0, 1, ...]           (1024 bits = 128 bytes)

Rule: value > 0 → 1, else → 0
```

**Trade-offs:**
- ✅ **97% smaller indexes** (4KB → 128 bytes)
- ✅ **Fastest HNSW traversal** — minimal I/O
- ✅ **Excellent for pre-filtering** large candidate sets
- ⚠️ **~90-95% recall** with single-stage (model dependent)
- ⚠️ **Requires two-stage retrieval** for high recall

**pgvector implementation:**

```sql
-- Create binary vector index
CREATE INDEX idx_products_binary ON products 
    USING hnsw ((binary_quantize(embedding)::bit(1024)) bit_hamming_ops)
    WITH (m = 16, ef_construction = 128);

-- Two-stage retrieval: Fast binary search → Precise reranking
WITH binary_candidates AS (
    -- Stage 1: Fast approximate search using Hamming distance
    -- Oversample 100x to ensure good candidates
    SELECT product_id, embedding
    FROM products
    ORDER BY binary_quantize(embedding)::bit(1024) <~> 
             binary_quantize('[...]'::vector)::bit(1024)
    LIMIT 1000  -- 100x oversample for 10 final results
)
-- Stage 2: Rerank with full precision vectors
SELECT product_id, product_description, price
FROM binary_candidates bc
JOIN products p USING (product_id)
ORDER BY p.embedding <=> '[...]'::vector
LIMIT 10;
```

**When to use binary:**
- ✅ 10M+ vectors where index size dominates costs
- ✅ Pre-filtering before expensive operations
- ✅ When you can implement two-stage retrieval
- ❌ Not recommended for < 1M vectors (halfvec is simpler)

---

## Quantization Decision Tree

```
                    ┌─────────────────────────────┐
                    │   How many vectors?         │
                    └─────────────────────────────┘
                               │
              ┌────────────────┼────────────────┐
              │                │                │
         < 100K          100K - 10M          > 10M
              │                │                │
              ▼                ▼                ▼
        ┌─────────┐      ┌─────────┐      ┌─────────┐
        │ float32 │      │ halfvec │      │  binary │
        │ (vector)│      │  (SQ)   │      │  (BQ)   │
        └─────────┘      └─────────┘      └─────────┘
              │                │                │
              ▼                ▼                ▼
         No index          Simple          Two-stage
         needed?          queries         + reranking
```

---

## Blaize Bazaar Recommendations

| Scale | Recommended Strategy | Index Size | Query Pattern |
|-------|---------------------|------------|---------------|
| **21K products** (current) | halfvec | ~42 MB | Direct query |
| **100K products** | halfvec | ~200 MB | Direct query |
| **1M products** | halfvec | ~2 GB | Direct query |
| **10M products** | binary + halfvec rerank | ~1.2 GB | Two-stage |

---

## Quick Reference: Distance Operators

| Type | L2 (Euclidean) | Cosine | Inner Product |
|------|----------------|--------|---------------|
| vector | `<->` | `<=>` | `<#>` |
| halfvec | `<->` | `<=>` | `<#>` |
| bit (binary) | — | — | `<~>` (Hamming) |

**Note:** Binary vectors use Hamming distance (`<~>`) which counts differing bits — conceptually different from cosine/L2 but effective for candidate retrieval.

In [ ]:
# ============================================================================
# Quantization Performance Comparison
# ============================================================================

def compare_quantization_performance(query: str, limit: int = 10):
    """Compare full precision vs halfvec vs binary quantization."""
    query_embedding = generate_embedding(query)
    
    with psycopg.connect(conn_string) as conn:
        with conn.cursor(row_factory=dict_row) as cur:
            print(f"\n🔍 Query: '{query}'")
            print("=" * 70)
            
            # Full precision search
            start = time.time()
            cur.execute("""
                SELECT "productId", product_description, 
                       1 - (embedding <=> %s::vector) as similarity
                FROM bedrock_integration.product_catalog
                WHERE quantity > 0
                ORDER BY embedding <=> %s::vector
                LIMIT %s
            """, (str(query_embedding), str(query_embedding), limit))
            full_results = cur.fetchall()
            full_time = (time.time() - start) * 1000
            
            print(f"\n📊 Full Precision (float32):")
            print(f"   Time: {full_time:.2f}ms")
            print(f"   Storage: 4KB per vector")
            print(f"   Top 3:")
            for i, r in enumerate(full_results[:3], 1):
                print(f"   {i}. {r['product_description'][:50]}... (sim: {r['similarity']:.3f})")
            
            # Note: halfvec and binary quantization require additional setup
            print(f"\n💡 halfvec (float16):")
            print(f"   Expected time: ~{full_time * 0.6:.2f}ms (40% faster)")
            print(f"   Storage: 2KB per vector (50% reduction)")
            print(f"   Recall: 98%+ (minimal accuracy loss)")
            
            print(f"\n⚡ Binary Quantization:")
            print(f"   Expected time: ~{full_time * 0.3:.2f}ms (70% faster)")
            print(f"   Storage: 128 bytes per vector (97% reduction)")
            print(f"   Recall: 95%+ (with two-stage retrieval)")

# Test
compare_quantization_performance("laptop for programming", limit=10)

# Section 4: Production Resilience Patterns

## Why Resilience Matters

**The challenge:** Multi-agent systems make many external API calls. When Bedrock experiences issues, your entire application can fail.

**Real-world scenario:**
- User query triggers 3 agents
- Each agent calls Bedrock for embeddings
- Bedrock is slow (30s timeout per call)
- Result: 3 × 30s = 90s of waiting before failure

**Solution:** Resilience patterns prevent cascading failures and provide graceful degradation.

---

## Circuit Breaker for Bedrock

**Problem:** Cascading failures when Bedrock is slow/unavailable

**Solution:** Circuit breaker pattern

**States:**
```
CLOSED → Normal operation, requests flow
OPEN → Failures detected, requests blocked (fail fast)
HALF_OPEN → Testing if service recovered
```

**How it works:**
1. **CLOSED state:** Requests flow normally to Bedrock
2. **After 5 failures:** Circuit opens, requests fail immediately (no Bedrock calls)
3. **After 60s timeout:** Circuit enters HALF_OPEN, tests one request
4. **If test succeeds:** Circuit closes, resume normal operation
5. **If test fails:** Circuit stays open, continue failing fast

**Benefits:**
- ✅ Prevents cascading failures across agents
- ✅ Faster error responses (no waiting for timeouts)
- ✅ Automatic recovery testing
- ✅ Protects Bedrock from overload during recovery

In [ ]:
# ============================================================================
# Circuit Breaker Pattern for Bedrock
# ============================================================================

class CircuitBreaker:
    """Simple circuit breaker for external service calls."""
    
    def __init__(self, failure_threshold: int = 5, timeout_seconds: int = 60):
        self.failure_threshold = failure_threshold
        self.timeout = timeout_seconds
        self.failures = 0
        self.last_failure_time = None
        self.state = "CLOSED"  # CLOSED, OPEN, HALF_OPEN
    
    def call(self, func, *args, **kwargs):
        """Execute function with circuit breaker protection."""
        if self.state == "OPEN":
            # Check if timeout has passed
            if time.time() - self.last_failure_time > self.timeout:
                self.state = "HALF_OPEN"
                print("🟡 Circuit breaker: HALF_OPEN (testing recovery)")
            else:
                raise Exception("Circuit breaker is OPEN - failing fast")
        
        try:
            result = func(*args, **kwargs)
            self.on_success()
            return result
        except Exception as e:
            self.on_failure()
            raise e
    
    def on_success(self):
        """Reset failure count on success."""
        self.failures = 0
        if self.state == "HALF_OPEN":
            self.state = "CLOSED"
            print("🟢 Circuit breaker: CLOSED (service recovered)")
    
    def on_failure(self):
        """Track failures and open circuit if threshold exceeded."""
        self.failures += 1
        self.last_failure_time = time.time()
        
        if self.failures >= self.failure_threshold:
            self.state = "OPEN"
            print(f"🔴 Circuit breaker: OPEN (failures: {self.failures})")

# Example usage
bedrock_breaker = CircuitBreaker(failure_threshold=3, timeout_seconds=30)

def safe_bedrock_call(text: str) -> List[float]:
    """Bedrock call with circuit breaker protection."""
    return bedrock_breaker.call(generate_embedding, text)

print("✅ Circuit breaker implemented")
print("   Failure threshold: 3")
print("   Timeout: 30 seconds")

## Exponential Backoff with Jitter

**Problem:** Retrying immediately can overwhelm recovering services

**Solution:** Exponential backoff + random jitter

**Formula:**
```python
delay = min(max_delay, base_delay * (2 ** attempt)) + random(0, jitter)

# Example with base=1s, max=60s, jitter=0.5s:
Attempt 1: 1s + random(0, 0.5s) = 1.0-1.5s
Attempt 2: 2s + random(0, 0.5s) = 2.0-2.5s  
Attempt 3: 4s + random(0, 0.5s) = 4.0-4.5s
Attempt 4: 8s + random(0, 0.5s) = 8.0-8.5s
```

**Why jitter matters:**
- Prevents "thundering herd" (all clients retrying at same time)
- Spreads load during recovery

In [ ]:
# ============================================================================
# Exponential Backoff with Jitter
# ============================================================================

import random
from typing import Callable, Any

def retry_with_backoff(
    func: Callable,
    max_retries: int = 3,
    base_delay: float = 1.0,
    max_delay: float = 60.0,
    jitter: float = 0.5,
    *args,
    **kwargs
) -> Any:
    """
    Retry function with exponential backoff and jitter.
    
    Args:
        func: Function to retry
        max_retries: Maximum number of retry attempts
        base_delay: Initial delay in seconds
        max_delay: Maximum delay cap in seconds
        jitter: Random jitter range (0 to jitter)
    """
    for attempt in range(max_retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == max_retries:
                print(f"❌ All {max_retries} retries failed")
                raise e
            
            # Calculate delay with exponential backoff + jitter
            delay = min(max_delay, base_delay * (2 ** attempt))
            delay += random.uniform(0, jitter)
            
            print(f"⚠️  Attempt {attempt + 1} failed: {str(e)[:50]}")
            print(f"   Retrying in {delay:.2f}s...")
            time.sleep(delay)

# Example usage
def resilient_embedding(text: str) -> List[float]:
    """Generate embedding with retry logic."""
    return retry_with_backoff(
        generate_embedding,
        max_retries=3,
        base_delay=1.0,
        text=text
    )

print("✅ Exponential backoff with jitter implemented")
print("   Max retries: 3")
print("   Backoff: 1s → 2s → 4s (+ random jitter)")

**Combined pattern:**
```python
# Use both circuit breaker AND retry with backoff
def super_resilient_call(text):
    """
    Combines circuit breaker (fail fast) with retry logic (transient failures).
    
    Flow:
    1. Circuit breaker checks if service is healthy
    2. If healthy, retry_with_backoff handles transient failures
    3. If circuit opens, fail immediately without retries
    """
    def wrapped_call():
        return retry_with_backoff(
            generate_embedding,
            max_retries=3,
            base_delay=1.0,
            text=text
        )
    
    return bedrock_breaker.call(wrapped_call)

# Usage
try:
    embedding = super_resilient_call("laptop for programming")
    print(f"✅ Success: {len(embedding)} dimensions")
except Exception as e:
    print(f"❌ Failed: {e}")
```

# Section 5: Observability & Cost Optimization

## Multi-Agent Cost Tracking

**Challenge:** Token costs across multiple agents add up fast

**Solution:** Track costs per agent, per session

**Metrics to track:**
```python
{
    "session_id": "uuid",
    "agent_name": "Pricing Specialist",
    "input_tokens": 150,
    "output_tokens": 300,
    "total_tokens": 450,
    "cost_usd": 0.0045,  # Claude Sonnet 4 pricing
    "latency_ms": 1250,
    "timestamp": "2024-11-19T10:30:00Z"
}
```

**Claude Sonnet 4 pricing:**
- Input: $3 per million tokens
- Output: $15 per million tokens

In [ ]:
# ============================================================================
# Multi-Agent Cost Tracking
# ============================================================================

class CostTracker:
    """Track token usage and costs across agents."""
    
    # Claude Sonnet 4 pricing (per million tokens)
    INPUT_COST_PER_MILLION = 3.00
    OUTPUT_COST_PER_MILLION = 15.00
    
    def __init__(self):
        self.session_costs = {}
    
    def track_usage(
        self,
        session_id: str,
        agent_name: str,
        input_tokens: int,
        output_tokens: int,
        latency_ms: float
    ):
        """Record token usage and calculate cost."""
        # Calculate costs
        input_cost = (input_tokens / 1_000_000) * self.INPUT_COST_PER_MILLION
        output_cost = (output_tokens / 1_000_000) * self.OUTPUT_COST_PER_MILLION
        total_cost = input_cost + output_cost
        
        # Store metrics
        if session_id not in self.session_costs:
            self.session_costs[session_id] = []
        
        self.session_costs[session_id].append({
            "agent_name": agent_name,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "cost_usd": total_cost,
            "latency_ms": latency_ms,
            "timestamp": datetime.utcnow().isoformat()
        })
    
    def get_session_summary(self, session_id: str) -> Dict:
        """Get cost summary for a session."""
        if session_id not in self.session_costs:
            return {"error": "Session not found"}
        
        calls = self.session_costs[session_id]
        
        return {
            "session_id": session_id,
            "total_calls": len(calls),
            "total_tokens": sum(c["total_tokens"] for c in calls),
            "total_cost_usd": sum(c["cost_usd"] for c in calls),
            "avg_latency_ms": sum(c["latency_ms"] for c in calls) / len(calls),
            "by_agent": self._aggregate_by_agent(calls)
        }
    
    def _aggregate_by_agent(self, calls: List[Dict]) -> Dict:
        """Aggregate costs by agent."""
        by_agent = {}
        for call in calls:
            agent = call["agent_name"]
            if agent not in by_agent:
                by_agent[agent] = {"calls": 0, "tokens": 0, "cost": 0.0}
            by_agent[agent]["calls"] += 1
            by_agent[agent]["tokens"] += call["total_tokens"]
            by_agent[agent]["cost"] += call["cost_usd"]
        return by_agent

# Example usage
tracker = CostTracker()

# Simulate tracking
session_id = "test-session-123"
tracker.track_usage(session_id, "Orchestrator", input_tokens=200, output_tokens=50, latency_ms=800)
tracker.track_usage(session_id, "Pricing Agent", input_tokens=500, output_tokens=300, latency_ms=1200)
tracker.track_usage(session_id, "Recommendation Agent", input_tokens=450, output_tokens=400, latency_ms=1500)

# Get summary
summary = tracker.get_session_summary(session_id)

print("✅ Cost tracking implemented")
print(f"\n📊 Session Summary:")
print(f"   Total calls: {summary['total_calls']}")
print(f"   Total tokens: {summary['total_tokens']:,}")
print(f"   Total cost: ${summary['total_cost_usd']:.4f}")
print(f"   Avg latency: {summary['avg_latency_ms']:.0f}ms")
print(f"\n   By Agent:")
for agent, metrics in summary['by_agent'].items():
    print(f"   • {agent}: {metrics['tokens']:,} tokens, ${metrics['cost']:.4f}")

## OpenTelemetry Tracing with Strands

**OpenTelemetry (OTel)** is integrated into Strands SDK for production monitoring.

**What Strands traces automatically:**
- Agent invocations (start/end times)
- Tool calls with parameters
- Token counts (input/output)
- Errors and exceptions
- Parent-child relationships (Orchestrator → Specialists)

**Trace structure:**
```
Orchestrator (span)
  ├─ Pricing Agent (span)
  │   ├─ get_category_price_analysis (span)
  │   └─ semantic_product_search (span)
  └─ Recommendation Agent (span)
      └─ get_trending_products (span)
```

**Blaize Bazaar exports traces to:**
- AWS X-Ray (distributed tracing)
- CloudWatch Logs (aggregation)
- Custom dashboards (Grafana/DataDog)

**Benefits:**
- ✅ End-to-end visibility across agents
- ✅ Identify bottlenecks (which agent is slow?)
- ✅ Debug multi-agent flows
- ✅ Cost attribution per trace

---

In [ ]:
# ============================================================================
# OpenTelemetry Integration Example
# ============================================================================

# Strands SDK automatically instruments agents with OTel
# This is what Blaize Bazaar uses in production

print("📊 OpenTelemetry Tracing")
print("\nStrands SDK provides automatic instrumentation:")
print("  • Agent invocations tracked as spans")
print("  • Tool calls with parameters logged")
print("  • Token usage and costs captured")
print("  • Parent-child relationships preserved")
print("\nBlaize Bazaar exports to:")
print("  • AWS X-Ray (distributed tracing)")
print("  • CloudWatch Logs (aggregation)")
print("  • Grafana dashboards (visualization)")
print("\nKey metrics monitored:")
print("  • P50/P95/P99 latencies per agent")
print("  • Error rates and failure modes")
print("  • Token usage trends over time")
print("  • Cost per conversation")

---

# Part 4 Complete!

## What You Learned

✅ **Aurora Session Management** - ACID guarantees + Multi-tenancy  
✅ **Hybrid Search with RRF** - Semantic + Keyword fusion  
✅ **Vector Quantization** - Binary & Scalar for 75-97% reduction  
✅ **Resilience Patterns** - Circuit breakers + Exponential backoff  
✅ **Observability** - Cost tracking + OpenTelemetry tracing

---

## Complete Workshop Journey

**✅ Part 1:** Semantic search with pgvector  
**✅ Part 2:** Custom tools with 96% token reduction  
**✅ Part 3:** Multi-agent orchestration  
**✅ Part 4:** Production patterns (you are here!)

---

## Continue Learning

**Blaize Bazaar Demo:**
- Full production implementation
- React frontend + FastAPI backend
- All patterns in action

**Resources:**
- [Strands SDK](https://strandsagents.com/)
- [pgvector Documentation](https://github.com/pgvector/pgvector)
- [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/)
- [OpenTelemetry](https://opentelemetry.io/)

**GitHub Repository:**
- [github.com/aws-samples/sample-dat406-build-agentic-ai-powered-search-apg](https://github.com/aws-samples/sample-dat406-build-agentic-ai-powered-search-apg)

---

<div style="text-align: center; color: #888; font-size: 0.9em; margin-top: 30px;">
Built for AWS re:Invent 2025 | DAT406 Workshop
</div>